# 10Web Armenia Jobs Scraping

**ATS:** BambooHR (`10web.bamboohr.com/careers`)  
**Method:** BambooHR exposes a public JSON API — `/careers/list` returns all openings, `/careers/{id}/detail` returns full job description. No auth required.  
**Outputs:** `data/raw/jobs/10web_jobs_raw.csv`, `data/processed/jobs/10web_jobs_standardized.csv`

In [1]:
from pathlib import Path, Path as P
import os

_nb = globals().get('__vsc_ipynb_file__') or globals().get('__file__', None)
if _nb:
    os.chdir(P(_nb).resolve().parent.parent.parent)
elif not (P.cwd() / 'data').exists():
    _r = P.cwd()
    for _ in range(4):
        if (_r / 'data').exists(): break
        _r = _r.parent
    os.chdir(_r)
assert (P.cwd() / 'data').exists(), f'Cannot find root from {P.cwd()}'
RAW_DIR  = P.cwd() / 'data/raw/jobs'
PROC_DIR = P.cwd() / 'data/processed/jobs'
print('Root:', P.cwd())

Root: /Users/lianaaghamalyan/thesis_data


In [2]:
import re, time, requests, pandas as pd
from bs4 import BeautifulSoup

BASE    = 'https://10web.bamboohr.com'
HEADERS = {'User-Agent': 'ThesisResearch/1.0 (Armenian IT curriculum alignment; academic use)',
           'Accept': 'application/json'}

def html_to_text(h):
    if not h: return ''
    t = BeautifulSoup(str(h), 'html.parser').get_text('\n', strip=True)
    return re.sub(r'\n{3,}', '\n\n', t).strip()

print(f'Base: {BASE}')

Base: https://10web.bamboohr.com


In [3]:
# Step 1: Get all job listings
resp = requests.get(f'{BASE}/careers/list', headers=HEADERS, timeout=20)
resp.raise_for_status()
all_jobs = resp.json().get('result', [])
print(f'Total jobs: {len(all_jobs)}')

# All jobs are Yerevan — filter defensively
def is_armenia(j):
    loc = j.get('location', {})
    city    = (loc.get('city') or '').lower()
    country = (loc.get('addressCountry') or '').lower()
    return 'yerevan' in city or 'armenia' in country or j.get('isRemote') is None

armenia_jobs = [j for j in all_jobs if is_armenia(j)]
print(f'Armenia/Yerevan jobs: {len(armenia_jobs)}')
for j in armenia_jobs:
    loc = j.get('location', {})
    print(f"  [{j['id']}] {j.get('jobOpeningName',''):<55} | {loc.get('city','')} | {j.get('departmentLabel','')}")

Total jobs: 5
Armenia/Yerevan jobs: 5
  [52] Senior Product Designer (UI/UX)                         | Yerevan | Product
  [57] Product Manager                                         | Yerevan | Product
  [59] Senior Software Engineer                                | Yerevan | Engineering
  [67] Senior Customer Care Specialist (Day Shift)             | Yerevan | Customer Care
  [68] Mid-Level Customer Care Specialist (Day Shift)          | Yerevan | Operations


In [4]:
# Step 2: Fetch full description for each job
records = []
for i, j in enumerate(armenia_jobs, 1):
    job_id = j['id']
    title  = j.get('jobOpeningName', '')
    print(f'[{i}/{len(armenia_jobs)}] {title}')

    try:
        r = requests.get(f'{BASE}/careers/{job_id}/detail', headers=HEADERS, timeout=15)
        r.raise_for_status()
        detail = r.json().get('result', r.json())
        opening = detail.get('jobOpening', detail)

        description_html = opening.get('description', '')
        full_text = html_to_text(description_html)
        print(f'  {len(full_text)} chars')

        loc = opening.get('location', j.get('location', {}))
        city    = loc.get('city', '')
        country = loc.get('addressCountry', 'Armenia')
        location = f'{city}, {country}' if city else country

        records.append({
            'source':          '10web',
            'source_type':     'company_portal',
            'source_url':      opening.get('jobOpeningShareUrl', f'{BASE}/careers/{job_id}'),
            'job_title':       title,
            'company_name':    '10Web',
            'location':        location,
            'department':      j.get('departmentLabel', ''),
            'employment_type': j.get('employmentStatusLabel', ''),
            'seniority_level': '',
            'industries':      'Technology / AI Website Builder',
            'posting_date':    '',
            'skills_tags':     '',
            'full_text':       full_text,
        })
    except Exception as e:
        print(f'  Error: {e}')
    time.sleep(0.5)

df = pd.DataFrame(records)
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(RAW_DIR  / '10web_jobs_raw.csv', index=False)
df.to_csv(PROC_DIR / '10web_jobs_standardized.csv', index=False)
print(f'\nSaved {len(df)} rows')
if len(df):
    print(df[['job_title', 'location', 'department', 'employment_type']].to_string())

[1/5] Senior Product Designer (UI/UX)
  2841 chars
[2/5] Product Manager
  3055 chars
[3/5] Senior Software Engineer
  1830 chars
[4/5] Senior Customer Care Specialist (Day Shift)
  2830 chars
[5/5] Mid-Level Customer Care Specialist (Day Shift)
  2413 chars

Saved 5 rows
                                        job_title          location     department employment_type
0                 Senior Product Designer (UI/UX)  Yerevan, Armenia        Product       Full-Time
1                                 Product Manager  Yerevan, Armenia        Product       Full-Time
2                        Senior Software Engineer  Yerevan, Armenia    Engineering       Full-Time
3     Senior Customer Care Specialist (Day Shift)  Yerevan, Armenia  Customer Care       Full-Time
4  Mid-Level Customer Care Specialist (Day Shift)  Yerevan, Armenia     Operations       Full-Time
